In [36]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/Documentation_template.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/README.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/utils/validate_submission.py
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_source2.tsv
/kaggl

# Preprocessing

This notebook prepares the raw business entity resolution datasets for candidate generation and matching.

The preprocessing pipeline will:

- Load the raw training and test datasets.
- Preserve the original source columns unchanged.
- Handle missing business names and addresses without dropping records.
- Create Unicode-safe normalized versions of business names and addresses.
- Standardize common formatting differences such as case, whitespace, punctuation, and `&` versus `and`.
- Create additional derived features useful for blocking and matching.
- Inspect the resulting preprocessing quality.

The preprocessing is designed for multilingual business records, including records from India and other countries.

No external data or entity lookup is used. All derived features are created only from the provided datasets.

## 1. Imports

In [37]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource")

TRAIN_DIR = ROOT / "dataset" / "train"
TEST_DIR = ROOT / "dataset" / "test"

print("Paths configured.")
print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

Paths configured.
Train directory: /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train
Test directory: /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test


## 2. Load Training Datasets

Only the training sources and ground truth are loaded.

Test data will remain completely separate and will be loaded fresh in `05_Predictions.ipynb`.

In [38]:
train_s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str
)

train_s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str
)

train_s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str
)

ground_truth = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("Training datasets loaded successfully.")
print(f"Train S1: {train_s1.shape}")
print(f"Train S2: {train_s2.shape}")
print(f"Train S3: {train_s3.shape}")
print(f"Ground truth: {ground_truth.shape}")

Training datasets loaded successfully.
Train S1: (2206821, 4)
Train S2: (5034616, 4)
Train S3: (5285603, 4)
Ground truth: (2206821, 2)


## 3. Verify Raw Columns

The original source columns must remain unchanged.

All preprocessing will be stored in new derived columns.

In [39]:
expected_columns = [
    "entity_id",
    "business_name",
    "business_address",
    "country"
]

for name, df in {
    "train_s1": train_s1,
    "train_s2": train_s2,
    "train_s3": train_s3,
}.items():
    assert list(df.columns) == expected_columns, (
        f"Unexpected columns in {name}"
    )

print("All training source datasets have the expected raw columns.")

All training source datasets have the expected raw columns.


## 4. Create Missing-Value Indicators

Missing records are not removed.

Instead, explicit indicators are created so later matching models can distinguish between:

- a genuinely empty field
- a field containing information

In [40]:
def add_missing_indicators(df):
    df["name_missing"] = (
        df["business_name"].isna()
        | df["business_name"].fillna("").str.strip().eq("")
    ).astype("int8")

    df["address_missing"] = (
        df["business_address"].isna()
        | df["business_address"].fillna("").str.strip().eq("")
    ).astype("int8")

    return df


for df in [train_s1, train_s2, train_s3]:
    add_missing_indicators(df)

print("Missing-value indicators created.")

Missing-value indicators created.


## 5. Define Unicode-Safe Text Normalization

The earlier EDA normalization was ASCII-based and could remove non-Latin scripts.

The final preprocessing preserves Unicode letters and numbers while standardizing:

- case
- Unicode representation
- ampersands
- punctuation
- whitespace

This is important for multilingual business records.

In [41]:
def normalize_unicode_text(text):
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    # Preserve Unicode scripts.
    text = unicodedata.normalize("NFKC", text)

    # Standardize ampersands.
    text = text.replace("&", " and ")

    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if char.isspace():
            cleaned.append(" ")
        elif category[0] in ("L", "N", "M"):
            cleaned.append(char)
        else:
            cleaned.append(" ")

    text = "".join(cleaned)

    return re.sub(r"\s+", " ", text).strip()


def normalize_name(text):
    return normalize_unicode_text(text)


def normalize_address(text):
    return normalize_unicode_text(text)


print("Unicode-safe normalization functions defined.")

Unicode-safe normalization functions defined.


## 6. Create Normalized Name and Address Fields

Normalized versions of business names and addresses are created for all three training sources.

The original raw fields remain unchanged.

In [42]:
for df in [train_s1, train_s2, train_s3]:
    df["name_norm"] = (
        df["business_name"]
        .fillna("")
        .map(normalize_name)
    )

    df["address_norm"] = (
        df["business_address"]
        .fillna("")
        .map(normalize_address)
    )

print("Normalized name and address fields created.")

Normalized name and address fields created.


## 7. Create Lightweight Text Features

Only inexpensive features required for later candidate generation are created here.

Length and token-count features provide basic information about the normalized fields without performing expensive character-level processing.

In [43]:
def add_text_features(df):
    df["name_len"] = (
        df["name_norm"].str.len().astype("int32")
    )

    df["address_len"] = (
        df["address_norm"].str.len().astype("int32")
    )

    df["name_token_count"] = (
        df["name_norm"]
        .str.count(" ")
        .add(1)
        .where(df["name_norm"].ne(""), 0)
        .astype("int16")
    )

    df["address_token_count"] = (
        df["address_norm"]
        .str.count(" ")
        .add(1)
        .where(df["address_norm"].ne(""), 0)
        .astype("int16")
    )

    return df


for df in [train_s1, train_s2, train_s3]:
    add_text_features(df)

print("Lightweight text features created.")

Lightweight text features created.


## 8. Create Initial Blocking Keys

Blocking keys provide reusable representations for the candidate-generation stage.

They are not final matching decisions.

The initial keys include:

- normalized business name
- first name token
- normalized address
- normalized country

In [44]:
def add_blocking_keys(df):
    df["name_key"] = df["name_norm"]

    df["name_first_token"] = (
        df["name_norm"]
        .str.split()
        .str[0]
        .fillna("")
    )

    df["address_key"] = df["address_norm"]

    df["country_key"] = (
        df["country"]
        .fillna("")
        .str.strip()
        .str.lower()
    )

    return df


for df in [train_s1, train_s2, train_s3]:
    add_blocking_keys(df)

print("Blocking keys created.")

Blocking keys created.


## 9. Inspect Preprocessed Records

A small sample is inspected to verify that:

- raw values remain available
- normalized values are populated
- missing indicators are correct
- derived features look reasonable

In [45]:
display(
    train_s1[
        [
            "entity_id",
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country",
            "name_missing",
            "address_missing",
            "name_len",
            "address_len",
            "name_token_count",
            "address_token_count",
        ]
    ].head(15)
)

,entity_id,business_name,name_norm,business_address,address_norm,country,name_missing,address_missing,name_len,address_len,name_token_count,address_token_count
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US,0,0,19,36,3,6
1,S1-773889195,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US,0,0,11,29,2,5
2,S1-377745466,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US,0,0,12,33,3,5
3,S1-133037285,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US,0,0,13,46,2,8
4,S1-755362802,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India,0,0,23,48,3,9
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US,0,0,26,31,4,5
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India,0,0,40,113,5,17
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US,0,0,17,41,3,7
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US,0,0,17,35,3,6
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US,0,0,26,56,3,10


## 10. Verify Unicode Preservation

The normalized representation preserves non-Latin Unicode characters.

This confirms that the final normalization does not reduce multilingual business names to ASCII-only text.

In [46]:
unicode_examples = [
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "குளோபல் பிசினஸ்",
    "શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ",
]

for text in unicode_examples:
    print("Original :", text)
    print("Normalized:", normalize_name(text))
    print()

Original : राम मार्केटिंग प्राइवेट लिमिटेड
Normalized: राम मार्केटिंग प्राइवेट लिमिटेड

Original : குளோபல் பிசினஸ்
Normalized: குளோபல் பிசினஸ்

Original : શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ
Normalized: શક્તિ અર્બન પ્રોડક્ટ્સ પ્રાઇવેટ



## 11. Final Preprocessing Validation

Verify that:

- no training rows were removed
- all three training sources contain the expected derived fields
- the ground truth remains unchanged
- no test data is loaded or processed

In [47]:
expected_rows = {
    "train_s1": 2_206_821,
    "train_s2": 5_034_616,
    "train_s3": 5_285_603,
    "ground_truth": 2_206_821,
}

assert len(train_s1) == expected_rows["train_s1"]
assert len(train_s2) == expected_rows["train_s2"]
assert len(train_s3) == expected_rows["train_s3"]
assert len(ground_truth) == expected_rows["ground_truth"]

required_features = [
    "name_missing",
    "address_missing",
    "name_norm",
    "address_norm",
    "name_len",
    "address_len",
    "name_token_count",
    "address_token_count",
    "name_key",
    "name_first_token",
    "address_key",
    "country_key",
]

for name, df in {
    "train_s1": train_s1,
    "train_s2": train_s2,
    "train_s3": train_s3,
}.items():
    missing = [
        col for col in required_features
        if col not in df.columns
    ]

    assert not missing, f"{name} missing columns: {missing}"


print("Preprocessing validation passed.")
print()
print(f"Train S1 rows: {len(train_s1):,}")
print(f"Train S2 rows: {len(train_s2):,}")
print(f"Train S3 rows: {len(train_s3):,}")
print(f"Ground truth rows: {len(ground_truth):,}")
print()
print("Training data only.")
print("No test data loaded or processed.")

Preprocessing validation passed.

Train S1 rows: 2,206,821
Train S2 rows: 5,034,616
Train S3 rows: 5,285,603
Ground truth rows: 2,206,821

Training data only.
No test data loaded or processed.


## 12. Save Processed Training Data

In [48]:
PROCESSED_DIR = Path("/kaggle/working/processed_train")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_s1.to_pickle(PROCESSED_DIR / "train_s1.pkl")
train_s2.to_pickle(PROCESSED_DIR / "train_s2.pkl")
train_s3.to_pickle(PROCESSED_DIR / "train_s3.pkl")
ground_truth.to_pickle(PROCESSED_DIR / "ground_truth.pkl")

print("Processed training data saved.")
print(PROCESSED_DIR)

Processed training data saved.
/kaggle/working/processed_train
